In [1]:
import argparse
from pathlib import Path

import numpy as np
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
# ---------------------------------------------------------------------------
# TPS computation
# ---------------------------------------------------------------------------

def compute_tps_matrix(adata, df_de, celltype_col="derived_class2_Dec2024",
                       donor_col="participant_id", de_celltype_col="cell_type",  de_gene_col="geneID",
                       de_logfc_col="logFC", cell_types=None,  min_cells_per_donor=0):
    
    """
    Compute TPS for each (cell type, donor) pair.

    Parameters
    ----------
    adata : anndata.AnnData
        Single-nucleus RNA-seq data with obs containing celltype_col and donor_col.
    df_de : pandas.DataFrame
        DE results with columns [de_celltype_col, de_gene_col, de_logfc_col].
    celltype_col : str
        Column in adata.obs with cell type labels.
    donor_col : str
        Column in adata.obs with donor / participant ID.
    de_celltype_col : str
        Column in df_de with cell type labels (matching celltype_col).
    de_gene_col : str
        Column with gene IDs (matching adata.var_names).
    de_logfc_col : str
        Column with log fold change values.
    cell_types : list or None
        List of cell types to process. If None, uses intersection of
        cell types present in adata and df_de.
    min_cells_per_donor : int
        If > 0, require at least this many nuclei per (cell type, donor)
        to include that cell type in TPS computation.

    Returns
    -------
    TPS_matrix : pandas.DataFrame
        DataFrame with index = cell types, columns = donors, values = TPS.
    """
    # Determine cell types to use
    adata_celltypes = adata.obs[celltype_col].unique()
    de_celltypes = df_de[de_celltype_col].unique()

    if cell_types is None:
        cell_types = sorted(set(adata_celltypes).intersection(de_celltypes))

    donors = adata.obs[donor_col].unique()
    TPS_matrix = pd.DataFrame(index=cell_types, columns=donors, dtype=float)

    # Optional: filter cell types by minimum cell count per donor
    if min_cells_per_donor > 0:
        counts = (
            adata.obs.groupby([celltype_col, donor_col])
            .size()
            .reset_index(name="n_cells")
        )
        counts_pivot = (
            counts.pivot(index=celltype_col, columns=donor_col, values="n_cells")
            .fillna(0)
        )
        keep_celltypes = counts_pivot[
            (counts_pivot >= min_cells_per_donor).all(axis=1)
        ].index
        cell_types = [ct for ct in cell_types if ct in keep_celltypes]

    print(f"Computing TPS for cell types: {cell_types}")
    print(f"Number of donors: {len(donors)}")

    # Iterate over cell types
    for cell_type in cell_types:
        print(f"  -> {cell_type}")

        # Subset AnnData to this cell type
        mask = adata.obs[celltype_col] == cell_type
        adata_sub = adata[mask]

        if adata_sub.n_obs == 0:
            print(f"    Skipping {cell_type}: no cells.")
            continue

        # Compute mean expression per gene per donor
        # to_df() returns a (cells x genes) DataFrame with var_names as columns
        expr_df = adata_sub.to_df()
        donor_series = adata_sub.obs[donor_col]
        mean_expr = expr_df.groupby(donor_series).mean()  # donors x genes

        # Subset DE to this cell type and index by gene ID
        de_sub = df_de[df_de[de_celltype_col] == cell_type].set_index(de_gene_col)

        # Restrict to genes present in both expression and DE
        common_genes = mean_expr.columns.intersection(de_sub.index)
        if len(common_genes) < 10:
            print(
                f"    Warning: {cell_type} has only {len(common_genes)} common genes, skipping."
            )
            continue

        mean_expr = mean_expr[common_genes]
        de_logfc = de_sub.loc[common_genes, de_logfc_col]

        # Baseline expression (mean across donors)
        baseline_expr = mean_expr.mean(axis=0)

        # Compute TPS per donor
        for donor in mean_expr.index:
            expr_profile = mean_expr.loc[donor]

            # Residualize by baseline expression and correlate with logFC
            try:
                r, _ = pearsonr(expr_profile - baseline_expr, de_logfc)
            except Exception:
                r = np.nan

            TPS_matrix.loc[cell_type, donor] = r

    return TPS_matrix


def compute_donor_tps(TPS_matrix, cell_types=None):
    """
    Compute donor-level TPS as mean across selected cell types.

    Parameters
    ----------
    TPS_matrix : pandas.DataFrame
        (cell types x donors) matrix of TPS values.
    cell_types : list or None
        Cell types to average over. If None, use all rows.

    Returns
    -------
    donor_TPS : pandas.Series
        Index = donors, values = mean TPS.
    """
    if cell_types is not None:
        missing = [ct for ct in cell_types if ct not in TPS_matrix.index]
        if missing:
            print(f"Warning: requested cell types not in TPS_matrix: {missing}")
        TPS_sub = TPS_matrix.loc[[ct for ct in cell_types if ct in TPS_matrix.index]]
    else:
        TPS_sub = TPS_matrix

    donor_TPS = TPS_sub.mean(axis=0)
    return donor_TPS


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------

def plot_tps_heatmap(TPS_matrix, out_path=None):
    """
    Plot a simple TPS heatmap (cell types x donors).

    Parameters
    ----------
    TPS_matrix : pandas.DataFrame
    out_path : pathlib.Path
        Output file path for PNG.
    """
    # Order donors and cell types by average TPS
    ordered_donors = TPS_matrix.mean(axis=0).sort_values().index
    ordered_celltypes = TPS_matrix.mean(axis=1).sort_values().index

    plt.figure(figsize=(12, 5))
    sns.heatmap(
        TPS_matrix.loc[ordered_celltypes, ordered_donors],
        cmap="coolwarm",
        center=0,
        xticklabels=True,
        yticklabels=True,
    )
    plt.xlabel("Donor (participant_id)")
    plt.ylabel("Cell type")
    plt.title("Transcriptomic Pathology Score (TPS)")
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"Saved TPS heatmap to: {out_path}")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser(
        description="Compute Transcriptomic Pathology Scores (TPS) "
        "from snRNA-seq AnnData and DE results."
    )
    p.add_argument(
        "--adata",
        required=True,
        help="Path to input .h5ad file with snRNA-seq data.",
    )
    p.add_argument(
        "--de_csv",
        required=True,
        help="Path to DE results CSV file.",
    )
    p.add_argument(
        "--outdir",
        required=True,
        help="Output directory for TPS CSVs and plots.",
    )
    p.add_argument(
        "--celltype_col",
        default="derived_class2_Dec2024",
        help="Column name in adata.obs with cell type labels.",
    )
    p.add_argument(
        "--donor_col",
        default="participant_id",
        help="Column name in adata.obs with donor IDs.",
    )
    p.add_argument(
        "--de_celltype_col",
        default="cell_type",
        help="Column name in DE table with cell type labels.",
    )
    p.add_argument(
        "--de_gene_col",
        default="geneID",
        help="Column name in DE table with gene identifiers.",
    )
    p.add_argument(
        "--de_logfc_col",
        default="logFC",
        help="Column name in DE table with log fold-change values.",
    )
    p.add_argument(
        "--min_cells_per_donor",
        type=int,
        default=0,
        help="Minimum number of cells per (cell type, donor) required "
        "to include that cell type (default: 0 = no filter).",
    )
    p.add_argument(
        "--cell_types",
        nargs="+",
        default=None,
        help="Optional explicit list of cell types to include "
        "(e.g. --cell_types Astro EN Endo IN Mural Myeloid OPC Oligo).",
    )
    p.add_argument(
        "--no_plot",
        action="store_true",
        help="If set, do not generate TPS heatmap.",
    )
    return p.parse_args()

In [3]:
## FDR Calculation Function ##

def fdr_per_celltype(dge_df):
    """Add FDR-adjusted p-values and significance per cell type."""
    dge_df['p.adjusted'] = None
    dge_df['significant_fdr'] = False
    
    for celltype, group in dge_df.groupby('assay'):
        rejected, pvals_corrected, _, _ = multipletests(group['p.value'], method='fdr_bh')
        dge_df.loc[group.index, 'p.adjusted'] = pvals_corrected
        dge_df.loc[group.index, 'significant_fdr'] = rejected
    
    return dge_df

In [4]:
## Import Dreamlet DGE ##

dreamlet_dge = pd.read_parquet('/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/res_meta.parquet')

dreamlet_metadata = pd.read_excel('/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/contrast_pairs.xlsx')

In [5]:
## Import Scz Adata ##

adata = sc.read_h5ad('/mnt/sdb/scz_meta_analysis_processed/anndata_objs/integrated_adata_annotated_w_celltypist_scanvi.h5ad')

In [6]:
adata.obs['Donor_Sample'] = adata.obs['Donor'].astype('str') + '_' + adata.obs['Sample Name'].astype('str')

In [7]:
adata.obs['psychad_celltypes'] = adata.obs['subclass_annotations_markers'].astype('str')

adata.obs.loc[adata.obs['subclass_annotations_markers'].str.contains('Neuron'), 'psychad_celltypes'] = 'EN'
adata.obs.loc[adata.obs['subclass_annotations_markers'].str.contains('GABAergic'), 'psychad_celltypes'] = 'IN'
adata.obs.loc[(adata.obs['subtype_cluster_annotations'] == 'OPC'), 'psychad_celltypes'] = 'OPC'
adata.obs.loc[(adata.obs['subtype_cluster_annotations'] == 'Oligodendrocyte'), 'psychad_celltypes'] = 'Oligo'
adata.obs.loc[(adata.obs['subtype_cluster_annotations'] == 'Astrocyte'), 'psychad_celltypes'] = 'Astro'

In [ ]:
## Prep Dreamlet DE DF ##

disease_list = ['AD', 'SCZ', 'DLBD', 'Vascular', 'Tauopathy', 'PD']
dx_code_list = ['m10x', 'm11x', 'm12x', 'm13x', 'm14x', 'm15x']

method_bool = (dreamlet_dge['method'] == 'FE') 
anno_bool = (dreamlet_dge['AnnoLevel'] == 'class')

tps_list = []

for disease, dx_code in zip(disease_list, dx_code_list):

    fe_class_dedf = None

    dx_bool = (dreamlet_dge['coef'] == dx_code)

    fe_class_dedf = dreamlet_dge[method_bool & anno_bool & dx_bool].copy()

    fe_class_dedf = fdr_per_celltype(fe_class_dedf)

    tps_dx = compute_tps_matrix(adata, fe_class_dedf, celltype_col="psychad_celltypes", 
                   donor_col="Donor_Sample",
                   de_celltype_col="assay", de_gene_col="ID", de_logfc_col="estimate", 
                   cell_types=None, min_cells_per_donor=0)

    tps_dx = tps_dx.T
    
    tps_dx.columns = [i+f'_{disease}' for i in tps_dx.columns]

    tps_list.append(tps_dx)

all_tps = pd.concat(tps_list, axis=1)

Computing TPS for cell types: ['Astro', 'EN', 'IN', 'OPC', 'Oligo']
Number of donors: 100
  -> Astro
  -> EN


In [ ]:
donor_genotype_dict = dict(zip(adata.obs['Donor_Sample'], adata.obs['Broad_Genotype']))

In [ ]:
all_tps['Broad_Genotype'] = all_tps.index.map(donor_genotype_dict)

In [ ]:
tps_melted = pd.melt(all_tps, id_vars=['Broad_Genotype'], var_name='CellType_Disease', value_name='TPS')

In [ ]:
tps_melted['CellType'] = tps_melted['CellType_Disease'].str.split('_').apply(lambda x: x[0])
tps_melted['Disease'] = tps_melted['CellType_Disease'].str.split('_').apply(lambda x: x[1])

In [ ]:
for celltype in tps_melted['CellType'].unique():
    temp_tps = tps_melted[
        (tps_melted['CellType'] == celltype) &
        (tps_melted['Disease'] == 'SCZ')
    ]
    
    ax = sns.boxplot(temp_tps, x='CellType', y='TPS', hue='Broad_Genotype')
    
    # Move legend to the right
    ax.legend(title='Broad_Genotype', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()  # prevents clipping
    plt.show()

In [ ]:
celltypes = tps_melted['CellType'].unique()
diseases = tps_melted['Disease'].unique()

for disease in diseases:
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes = axes.flatten()
    
    for i, celltype in enumerate(celltypes):
        ax = axes[i]
        
        temp_tps = tps_melted[
            (tps_melted['CellType'] == celltype) &
            (tps_melted['Disease'] == disease)
        ]
    
        sns.violinplot(
            data=temp_tps,
            x='CellType',
            y='TPS',
            hue='Broad_Genotype',
            ax=ax,
            legend=False
        )
        
        ax.set_title(celltype)
    
    # single legend for the figure
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, bbox_to_anchor=(1.02, 0.5), loc='center left')

    fig.suptitle(disease, fontsize=16)

    plt.tight_layout()
    plt.show()

In [ ]:
# separate Broad_Genotype from feature columns
meta_col = "Broad_Genotype"
feature_cols = [c for c in all_tps.columns if c != meta_col]

# build a helper dataframe to sort columns
col_df = pd.DataFrame({"col": feature_cols, "celltype": [c.split("_")[0] for c in feature_cols],
                       "disease": [c.split("_")[1] for c in feature_cols]})

# define desired order (optional, otherwise alphabetical)
disease_order = ["AD", "SCZ", "Tauopathy", "PD", 'DLBD', 'Vascular']
celltype_order = ["Astro", "EN", "IN", "OPC", "Oligo"]

col_df["disease"] = pd.Categorical(col_df["disease"], categories=disease_order, ordered=True)
col_df["celltype"] = pd.Categorical(col_df["celltype"], categories=celltype_order, ordered=True)

# sort
col_df = col_df.sort_values(["celltype", "disease"])

# reorder dataframe
all_tps_heatmap = all_tps[col_df["col"].tolist() + [meta_col]].copy()

all_tps_heatmap = all_tps_heatmap.sort_values(by='Broad_Genotype')

del all_tps_heatmap['Broad_Genotype']

In [ ]:
sns.heatmap(all_tps_heatmap)

In [ ]:
## Experimenet with Heatmap ##

astro_tps = all_tps[[i for i in all_tps.columns if 'Astro' in i] + ['Broad_Genotype']].sort_values(by='Broad_Genotype').copy()

astro_tps = astro_tps.groupby('Broad_Genotype').mean()

sns.heatmap(astro_tps)

In [ ]:
## Write TPS To File ##

all_tps.to_csv('/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/tps_files/all_tps.csv')

tps_melted.to_csv('/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/tps_files/tps_melted.csv')